# Regime lab

A regime detector is a signal like any other: it has to prove it can find regimes that exist (synthetic bench), then
that its state predicts which strategy earns more next on real data (regime card, with a timing placebo), and only
then may it drive an overlay, which must beat its own benchmarks.

1. Signals from the library
2. The synthetic bench: known regimes, scored detectors, stability under noise
3. Regime cards on real data (NDX, H14's market)
4. An overlay and its benchmarks
5. Your idea

Change a setting and re-run the cells below it; features are cached, so only what changed recomputes.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quantlab.data import align
from quantlab.signals import library as lib
from quantlab.engine.components import Targets, apply_signal
from quantlab.engine.backtest import run_backtest, Sizer, EngineConfig
from quantlab.analytics.performance import metrics
from quantlab.analytics.conditional import by_regime, after_changes
from quantlab.regimes.models import FeatureBlendRegime, PercentileRegime, VolatilityRegime, HMMRegime, labels
from quantlab.regimes.simulate import RegimeProcess, simulate, simulate_paths
from quantlab.regimes.bench import bench, sweep
from quantlab.validation.regime_card import regime_card
from quantlab.overlays.overlay import RegimeOverlay, benchmarks

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

## 1. Signals

Every library signal states its hypothesis, expected sign and horizon, and is on the forecast scale (sd about 1, capped at ±2).

In [ ]:
bars = pd.read_csv("../data/research_cache/h14/NDX.csv", index_col=0, parse_dates=True)
bars.columns = [c.lower() for c in bars.columns]
panel = align({"NDX": bars})
returns = bars["close"].pct_change().to_frame("NDX")

pd.DataFrame({name: {"hypothesis": f().hypothesis, "positive means": f().sign, "horizon": f().horizon} for name, f in lib.CATALOG.items()}).T

In [ ]:
recent = bars.loc["2018":]
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(recent["close"], color="black", lw=0.8); axes[0].set_yscale("log"); axes[0].set_title("NDX")
for signal in (lib.ewmac(16, 64), lib.h14_trend(), lib.zscore_reversion(40)):
    axes[1].plot(signal(bars).loc["2018":], lw=0.8, label=signal.name)
axes[1].axhline(0, color="grey", lw=0.5); axes[1].legend(); axes[1].set_title("forecasts")
plt.tight_layout(); plt.show()

## 2. The synthetic bench

Simulated markets with TREND, RANGE and SHOCK episodes whose truth is known. A detector that cannot find these
should not be trusted on real data. Edit `RegimeProcess(...)` to change episode lengths, drift, noise or add a
structural break (`break_at=0.5`).

In [ ]:
process = RegimeProcess()
sample, truth = simulate(process, days=6 * 252, seed=1)
colours = {"TREND": "tab:green", "RANGE": "tab:orange", "SHOCK": "tab:red"}
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(sample["close"], color="black", lw=0.8)
for start, end in zip(truth.index[truth.ne(truth.shift())], list(truth.index[truth.ne(truth.shift())][1:]) + [truth.index[-1]]):
    ax.axvspan(start, end, color=colours[truth[start]], alpha=0.15, lw=0)
ax.set_title("one simulated path (green trend, orange range, red shock)"); plt.show()

In [ ]:
two_state = {"TREND": "TREND", "RANGE": "RANGE"}
detectors = {
    "H14 feature blend": (FeatureBlendRegime(), {"TREND": "TREND", "RANGE": "RANGE", "TRANSITION": None}),
    "efficiency ratio": (PercentileRegime("er"), two_state),
    "|slope t|": (PercentileRegime("abs_slope_t"), two_state),
    "ADX": (PercentileRegime("adx"), two_state),
    "|slope t|, 3-bar confirm": (PercentileRegime("abs_slope_t", confirm=3), two_state),
    "HMM (calm = trend)": (HMMRegime(), {"CALM": "TREND", "VOLATILE": "RANGE"}),
}
paths = simulate_paths(process, years=15, seeds=range(4))
scores = bench(detectors, paths)
scores[["coverage", "balanced accuracy", "balanced accuracy (std across paths)", "recall TREND", "recall RANGE",
        "median lag (bars)", "episodes missed", "false switches a year", "true switches a year"]].round(3)

In [ ]:
stability = sweep({k: v for k, v in detectors.items() if not k.startswith("HMM")}, process, "noise", [0.5, 1.0, 1.5, 2.0], years=10, seeds=range(3))
stability.plot(marker="o", figsize=(8, 4), title="balanced accuracy as noise grows (0.5 = chance)"); plt.axhline(0.5, color="grey", lw=0.5); plt.show()

## 3. Regime cards on real data

Does the state at the close of t predict whether trend beats mean reversion over the next 20 bars? The edge has a
block-bootstrap interval and a placebo: the same labels shifted in time. Pass marks: interval above zero and placebo
p ≤ 0.05, with at least 20 spells per state.

**Several cards are run here, so count the trials:** with six detectors, one passing at p ≈ 0.04 is roughly what
chance gives. Pre-register the detector before reading its card.

In [ ]:
sized = dict(costs_bps=1.5, sizer=Sizer(0.10, span=20, cap=2.0), config=EngineConfig(band=0.02, max_position=2.0))
trend_pnl = run_backtest(Targets(apply_signal(lib.h14_trend(), panel)), returns, **sized).pnl
reversion_pnl = run_backtest(Targets(apply_signal(lib.zscore_reversion(40), panel)), returns, **sized).pnl

candidates = {
    "H14 feature blend": (FeatureBlendRegime(), "TREND", "RANGE"),
    "efficiency ratio": (PercentileRegime("er"), "TREND", "RANGE"),
    "|slope t|": (PercentileRegime("abs_slope_t"), "TREND", "RANGE"),
    "ADX": (PercentileRegime("adx"), "TREND", "RANGE"),
    "volatility (calm = trend)": (VolatilityRegime(), "CALM", "STRESSED"),
    "HMM (calm = trend)": (HMMRegime(), "CALM", "VOLATILE"),
}
cards = {name: regime_card(model(bars), trend_pnl, reversion_pnl, state_a=a, state_b=b, name=name, bars=bars,
                           redundancy_signal=lib.h14_trend()(bars)) for name, (model, a, b) in candidates.items()}
pd.DataFrame({name: {"verdict": c.verdict, "edge (20 bars)": round(c.details["edge"], 4),
                     "interval": tuple(round(x, 4) for x in c.details["interval"]), "placebo p": round(c.details["placebo_p"], 3),
                     "switches a year": round(c.details["switches"], 1), "corr with |trend|": round(c.details["redundancy"], 2)}
              for name, c in cards.items()}).T

In [ ]:
pick = "HMM (calm = trend)"
card = cards[pick]
print(card.verdict, "-", card.reason)
display(card.behaviour.round(2))
plt.hist(card.details["placebo"], bins=40, color="grey"); plt.axvline(card.details["edge"], color="red")
plt.title(f"{pick}: edge vs shifted-label placebo"); plt.show()

## 4. An overlay and its benchmarks

Any signals, any regime model, weights per state. Soft probabilities blend; `hard=True` switches. The benchmarks
come automatically: static blend, hard switch, exposure-matched static, each signal alone. An overlay that does not
beat the static blend and the exposure-matched static is not adding information through its regimes.

As committed, this example (efficiency-ratio blend of H14's trend and a 40-day reversion) loses to its static blend,
its exposure-matched static and trend alone: the same finding as H14. Its regimes cost turnover without adding
information.

The by-regime table at the end is descriptive: the label and the P&L overlap in time, so a trend label "explains"
trend P&L after the fact. Whether a label *predicts* is the regime card's question (section 3).

In [ ]:
overlay = RegimeOverlay(
    signals={"trend": lib.h14_trend(), "reversion": lib.zscore_reversion(40)},
    regime=PercentileRegime("er", soft=True),
    weights={"TREND": {"trend": 1.0}, "RANGE": {"trend": 0.3, "reversion": 0.7}},
    cap=1.0,
)
primary = overlay(panel)
books = {"overlay": primary, **benchmarks(overlay, panel, primary=primary)}
results = {name: run_backtest(t, returns, **sized) for name, t in books.items()}
pd.DataFrame({name: metrics(r) for name, r in results.items()}).T[["Sharpe", "annual return", "volatility", "max drawdown", "turnover", "time invested"]].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for name, result in results.items():
    ax.plot(result.pnl.cumsum(), lw=1.4 if name == "overlay" else 0.8, label=name)
ax.set_title("cumulative return (sum of daily returns)"); ax.legend(); plt.show()

In [ ]:
state = labels(PercentileRegime("er")(bars))
display(by_regime({name: r.pnl for name, r in results.items()}, state).round(2))
after_changes({name: r.pnl for name, r in results.items()}, state).round(1)

## 5. Your idea

Copy a detector or an overlay from above and change it. A regime model is any callable `bars -> DataFrame of
state probabilities`; a feature for `PercentileRegime` can be any `bars -> Series`, e.g.:

```python
my_feature = lambda b: b["close"].pct_change().rolling(20).std() / b["close"].pct_change().rolling(120).std()
mine = PercentileRegime(my_feature, state_names=("CALM", "STRESSED"))
```

Write the hypothesis down (`scripts/research/new_hypothesis.py`) before you look at its card.

In [ ]:
mine = PercentileRegime("abs_slope_t", horizon=60, confirm=3)
bench({"mine": (mine, two_state)}, paths)[["balanced accuracy", "median lag (bars)", "false switches a year"]]